# Week 1 — Dataset Understanding and Exploratory Data Analysis

This notebook documents reproducible inspection only. It does **not** modify raw data, engineer features, preprocess data, or train models.


## 1. Setup and data loading

The large training table is loaded below for interactive exploration. For the complete memory-safe full scan and all report generation, run `python ../scripts/week1_analysis.py` from this notebook directory or `python scripts/week1_analysis.py` from the repository root.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
FIGURES = ROOT / 'reports' / 'figures'
sns.set_theme(style='whitegrid')


In [ ]:
building = pd.read_csv(RAW / 'building_metadata.csv')
weather = pd.read_csv(RAW / 'weather_train.csv', parse_dates=['timestamp'])
# Use the full table when memory permits; the report script always uses chunks.
train = pd.read_csv(RAW / 'train.csv', parse_dates=['timestamp'])


## 2. Reusable inspection helpers


In [ ]:
def dataset_overview(frames):
    return pd.DataFrame({name: {'rows': len(df), 'columns': df.shape[1], 'memory_mb': df.memory_usage(deep=True).sum()/2**20, 'duplicate_rows': df.duplicated().sum()} for name, df in frames.items()}).T

def missing_profile(df):
    out = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': df.isna().mean()*100, 'dtype': df.dtypes.astype(str), 'unique': df.nunique(dropna=False)})
    return out.sort_values('missing_pct', ascending=False)


In [ ]:
frames = {'train': train, 'building_metadata': building, 'weather_train': weather}
display(dataset_overview(frames))
for name, frame in frames.items():
    print(f'\n{name}')
    display(missing_profile(frame))


## 3. Relationship and key validation

Merge order: training observations → building metadata → weather. Left joins preserve every labelled observation.


In [ ]:
assert building['building_id'].is_unique, 'Metadata building_id must be unique'
assert not weather.duplicated(['site_id', 'timestamp']).any(), 'Weather key must be unique'
assert not train.duplicated(['building_id', 'meter', 'timestamp']).any(), 'Training observation key must be unique'

merged = train.merge(building, on='building_id', how='left', validate='many_to_one', indicator='_metadata_match')
merged = merged.merge(weather, on=['site_id', 'timestamp'], how='left', validate='many_to_one', indicator='_weather_match')
print(merged['_metadata_match'].value_counts(dropna=False))
print(merged['_weather_match'].value_counts(dropna=False))


## 4. Target, zero values, distributions and outliers


In [ ]:
display(train['meter_reading'].describe(percentiles=[.01,.25,.5,.75,.95,.99,.999]))
print('Zero percentage:', train['meter_reading'].eq(0).mean()*100)
print('Skewness:', train['meter_reading'].skew())
sns.histplot(np.log1p(train['meter_reading']), bins=80); plt.title('Target distribution'); plt.xlabel('log1p(meter_reading)');


In [ ]:
plot_sample = merged.sample(min(200_000, len(merged)), random_state=42).copy()
plot_sample['log_target'] = np.log1p(plot_sample['meter_reading'])
sns.boxplot(data=plot_sample, x='meter', y='log_target', showfliers=False); plt.title('Target by meter');


## 5. Building, meter, site and weather analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_sample.groupby('meter')['meter_reading'].median().plot.bar(ax=axes[0], title='Meter median')
plot_sample.groupby('site_id')['meter_reading'].median().plot.bar(ax=axes[1], title='Site median')
plot_sample.groupby('primary_use')['meter_reading'].median().sort_values().plot.barh(ax=axes[2], title='Primary-use median')
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=plot_sample, x='air_temperature', y='log_target', hue='meter', alpha=.15, s=10, ax=axes[0])
sns.scatterplot(data=plot_sample, x='square_feet', y='log_target', hue='meter', alpha=.15, s=10, ax=axes[1]); axes[1].set_xscale('log'); plt.tight_layout()


## 6. Time coverage and trends


In [ ]:
print(train['timestamp'].min(), train['timestamp'].max())
time_df = train.set_index('timestamp')['meter_reading']
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
time_df.groupby(time_df.index.hour).mean().plot(ax=axes[0], title='Hourly mean')
time_df.resample('D').mean().plot(ax=axes[1], title='Daily mean')
time_df.resample('MS').mean().plot.bar(ax=axes[2], title='Monthly / seasonal mean')
plt.tight_layout()


## 7. Correlation and skewness


In [ ]:
numeric = plot_sample.select_dtypes(include='number')
display(numeric.skew().sort_values(ascending=False).to_frame('skewness'))
plt.figure(figsize=(12, 9)); sns.heatmap(numeric.corr(method='spearman'), cmap='coolwarm', center=0); plt.title('Spearman correlation matrix');


## 8. Conclusions and Week 2 recommendations

The dataset is suitable for time-aware regression. Key issues are missing weather/metadata, extreme right skew, zeros, outliers, high-cardinality categorical IDs, meter/site heterogeneity, memory usage, and temporal leakage.

Week 2 should implement—but this notebook does not implement—chronological splitting, training-only imputation/encoding/scaling, calendar and strictly lagged features, and baseline comparisons using RMSLE plus MAE/RMSE.
